# Optimization Algorithms: Complete Comparison

**Goal**: Understand and compare gradient-based optimization algorithms used in deep learning.

---

## Table of Contents

1. [Introduction](#introduction)
2. [Gradient Descent Variants](#gd-variants)
3. [Momentum-Based Methods](#momentum)
4. [Adaptive Learning Rate Methods](#adaptive)
5. [Modern Optimizers](#modern)
6. [Practical Comparison](#comparison)
7. [Hyperparameter Sensitivity](#sensitivity)
8. [Best Practices](#best-practices)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from IPython.display import HTML

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Random seed
np.random.seed(42)

## 1. Introduction <a name="introduction"></a>

### The Optimization Problem

In machine learning, we want to minimize a loss function:

$$
\min_{\theta} \mathcal{L}(\theta)
$$

Where:
- $\theta$: Model parameters
- $\mathcal{L}$: Loss function

### Challenges

1. **High dimensionality**: Millions of parameters
2. **Non-convexity**: Multiple local minima
3. **Ill-conditioning**: Different curvatures in different directions
4. **Noisy gradients**: Stochastic mini-batches

### Test Functions

We'll use these functions to visualize optimizer behavior:

In [ ]:
class TestFunctions:
    """Collection of test functions for optimization."""
    
    @staticmethod
    def sphere(x):
        """Simple convex function: f(x) = ||x||^2"""
        return np.sum(x**2)
    
    @staticmethod
    def sphere_grad(x):
        return 2 * x
    
    @staticmethod
    def rosenbrock(x):
        """Rosenbrock function (non-convex, narrow valley)."""
        return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2
    
    @staticmethod
    def rosenbrock_grad(x):
        dx0 = -2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2)
        dx1 = 200 * (x[1] - x[0]**2)
        return np.array([dx0, dx1])
    
    @staticmethod
    def beale(x):
        """Beale function (non-convex)."""
        return ((1.5 - x[0] + x[0]*x[1])**2 + 
                (2.25 - x[0] + x[0]*x[1]**2)**2 + 
                (2.625 - x[0] + x[0]*x[1]**3)**2)
    
    @staticmethod
    def beale_grad(x):
        t1 = 1.5 - x[0] + x[0]*x[1]
        t2 = 2.25 - x[0] + x[0]*x[1]**2
        t3 = 2.625 - x[0] + x[0]*x[1]**3
        
        dx0 = (2*t1*(-1 + x[1]) + 
               2*t2*(-1 + x[1]**2) + 
               2*t3*(-1 + x[1]**3))
        dx1 = (2*t1*x[0] + 
               2*t2*x[0]*2*x[1] + 
               2*t3*x[0]*3*x[1]**2)
        return np.array([dx0, dx1])
    
    @staticmethod
    def rastrigin(x, A=10):
        """Rastrigin function (highly multimodal)."""
        n = len(x)
        return A*n + np.sum(x**2 - A*np.cos(2*np.pi*x))
    
    @staticmethod
    def rastrigin_grad(x, A=10):
        return 2*x + 2*np.pi*A*np.sin(2*np.pi*x)

### Visualize Test Functions

In [ ]:
def plot_function_2d(func, xlim, ylim, title, levels=30):
    """Plot 2D function contours."""
    x = np.linspace(xlim[0], xlim[1], 200)
    y = np.linspace(ylim[0], ylim[1], 200)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = func(np.array([X[i, j], Y[i, j]]))
    
    plt.figure(figsize=(10, 8))
    plt.contourf(X, Y, Z, levels=levels, cmap='viridis', alpha=0.7)
    plt.colorbar(label='Function Value')
    plt.contour(X, Y, Z, levels=levels, colors='black', alpha=0.3, linewidths=0.5)
    plt.xlabel('x₁')
    plt.ylabel('x₂')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    return X, Y, Z

# Plot all test functions
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Sphere
plt.subplot(2, 2, 1)
plot_function_2d(TestFunctions.sphere, [-2, 2], [-2, 2], 'Sphere Function (Convex)')

# Rosenbrock
plt.subplot(2, 2, 2)
plot_function_2d(TestFunctions.rosenbrock, [-2, 2], [-1, 3], 'Rosenbrock Function (Narrow Valley)')

# Beale
plt.subplot(2, 2, 3)
plot_function_2d(TestFunctions.beale, [-4.5, 4.5], [-4.5, 4.5], 'Beale Function (Non-Convex)')

# Rastrigin
plt.subplot(2, 2, 4)
plot_function_2d(TestFunctions.rastrigin, [-5, 5], [-5, 5], 'Rastrigin Function (Multimodal)', levels=50)

plt.tight_layout()
plt.show()

---

## 2. Gradient Descent Variants <a name="gd-variants"></a>

### 2.1 Batch Gradient Descent (BGD)

Compute gradient using **all** training examples:

$$
\theta_{t+1} = \theta_t - \eta \nabla_{\theta} \mathcal{L}(\theta_t)
$$

**Pros**: Stable convergence, exact gradient
**Cons**: Slow for large datasets

### 2.2 Stochastic Gradient Descent (SGD)

Compute gradient using **one** example:

$$
\theta_{t+1} = \theta_t - \eta \nabla_{\theta} \mathcal{L}_i(\theta_t)
$$

**Pros**: Fast updates, can escape local minima
**Cons**: Noisy, unstable

### 2.3 Mini-Batch Gradient Descent

Compromise: Use a **small batch** (e.g., 32, 64, 256):

$$
\theta_{t+1} = \theta_t - \eta \frac{1}{|B|} \sum_{i \in B} \nabla_{\theta} \mathcal{L}_i(\theta_t)
$$

**This is the standard in deep learning!**

### Implementation

In [ ]:
class SGD:
    """Stochastic Gradient Descent."""
    
    def __init__(self, lr=0.01):
        self.lr = lr
        
    def update(self, params, grads):
        """Update parameters."""
        return params - self.lr * grads
    
    def reset(self):
        """Reset optimizer state."""
        pass

---

## 3. Momentum-Based Methods <a name="momentum"></a>

### 3.1 Momentum

**Idea**: Accumulate past gradients to smooth updates

$$
\begin{align}
v_t &= \beta v_{t-1} + \nabla_{\theta} \mathcal{L}(\theta_{t-1}) \\
\theta_t &= \theta_{t-1} - \eta v_t
\end{align}
$$

Where:
- $v_t$: Velocity (exponential moving average of gradients)
- $\beta$: Momentum coefficient (typically 0.9)

**Physical Analogy**: Ball rolling down a hill
- Accumulates speed in consistent directions
- Dampens oscillations

**Benefits**:
- Faster convergence in ravines
- Reduces oscillations
- Can escape shallow local minima

### 3.2 Nesterov Accelerated Gradient (NAG)

**Idea**: "Look ahead" before computing gradient

$$
\begin{align}
v_t &= \beta v_{t-1} + \nabla_{\theta} \mathcal{L}(\theta_{t-1} - \beta v_{t-1}) \\
\theta_t &= \theta_{t-1} - \eta v_t
\end{align}
$$

**Difference**: Compute gradient at **anticipated position**

**Benefit**: Better anticipates changes, reduces overshooting

### Implementation

In [ ]:
class Momentum:
    """SGD with Momentum."""
    
    def __init__(self, lr=0.01, beta=0.9):
        self.lr = lr
        self.beta = beta
        self.v = None
        
    def update(self, params, grads):
        """Update parameters with momentum."""
        if self.v is None:
            self.v = np.zeros_like(params)
        
        # Update velocity
        self.v = self.beta * self.v + grads
        
        # Update parameters
        return params - self.lr * self.v
    
    def reset(self):
        self.v = None


class Nesterov:
    """Nesterov Accelerated Gradient."""
    
    def __init__(self, lr=0.01, beta=0.9):
        self.lr = lr
        self.beta = beta
        self.v = None
        
    def update(self, params, grad_func):
        """Update parameters with Nesterov momentum.
        
        Note: Requires gradient function, not precomputed gradient.
        """
        if self.v is None:
            self.v = np.zeros_like(params)
        
        # Look ahead
        params_ahead = params - self.beta * self.v
        
        # Compute gradient at look-ahead position
        grads = grad_func(params_ahead)
        
        # Update velocity
        self.v = self.beta * self.v + grads
        
        # Update parameters
        return params - self.lr * self.v
    
    def reset(self):
        self.v = None

---

## 4. Adaptive Learning Rate Methods <a name="adaptive"></a>

### 4.1 AdaGrad

**Idea**: Adapt learning rate for each parameter based on historical gradients

$$
\begin{align}
G_t &= G_{t-1} + g_t \odot g_t \\
\theta_t &= \theta_{t-1} - \frac{\eta}{\sqrt{G_t + \epsilon}} \odot g_t
\end{align}
$$

Where:
- $G_t$: Sum of squared gradients (element-wise)
- $g_t = \nabla \mathcal{L}(\theta_{t-1})$
- $\odot$: Element-wise multiplication
- $\epsilon$: Small constant for numerical stability (e.g., 1e-8)

**Effect**: 
- Large gradients → Small learning rate
- Small gradients → Large learning rate

**Problem**: Learning rate monotonically decreases → can stop too early

### 4.2 RMSprop

**Fix**: Use exponential moving average instead of sum

$$
\begin{align}
E[g^2]_t &= \beta E[g^2]_{t-1} + (1-\beta) g_t^2 \\
\theta_t &= \theta_{t-1} - \frac{\eta}{\sqrt{E[g^2]_t + \epsilon}} g_t
\end{align}
$$

**Benefit**: Learning rate doesn't vanish

### 4.3 Adam (Adaptive Moment Estimation)

**Combines**: Momentum + RMSprop

$$
\begin{align}
m_t &= \beta_1 m_{t-1} + (1-\beta_1) g_t \quad \text{(1st moment, momentum)} \\
v_t &= \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \quad \text{(2nd moment, RMSprop)} \\
\hat{m}_t &= \frac{m_t}{1 - \beta_1^t} \quad \text{(bias correction)} \\
\hat{v}_t &= \frac{v_t}{1 - \beta_2^t} \quad \text{(bias correction)} \\
\theta_t &= \theta_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t
\end{align}
$$

**Default hyperparameters**:
- $\beta_1 = 0.9$ (momentum)
- $\beta_2 = 0.999$ (RMSprop)
- $\eta = 0.001$ (learning rate)
- $\epsilon = 10^{-8}$

**Why bias correction?** 
- $m_0 = 0, v_0 = 0$ → biased towards zero initially
- Correction compensates for this

### Implementation

In [ ]:
class AdaGrad:
    """AdaGrad optimizer."""
    
    def __init__(self, lr=0.01, epsilon=1e-8):
        self.lr = lr
        self.epsilon = epsilon
        self.G = None
        
    def update(self, params, grads):
        if self.G is None:
            self.G = np.zeros_like(params)
        
        # Accumulate squared gradients
        self.G += grads ** 2
        
        # Update parameters
        return params - self.lr * grads / (np.sqrt(self.G) + self.epsilon)
    
    def reset(self):
        self.G = None


class RMSprop:
    """RMSprop optimizer."""
    
    def __init__(self, lr=0.01, beta=0.9, epsilon=1e-8):
        self.lr = lr
        self.beta = beta
        self.epsilon = epsilon
        self.E_g2 = None
        
    def update(self, params, grads):
        if self.E_g2 is None:
            self.E_g2 = np.zeros_like(params)
        
        # Update moving average of squared gradients
        self.E_g2 = self.beta * self.E_g2 + (1 - self.beta) * grads ** 2
        
        # Update parameters
        return params - self.lr * grads / (np.sqrt(self.E_g2) + self.epsilon)
    
    def reset(self):
        self.E_g2 = None


class Adam:
    """Adam optimizer."""
    
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0
        
    def update(self, params, grads):
        if self.m is None:
            self.m = np.zeros_like(params)
            self.v = np.zeros_like(params)
        
        self.t += 1
        
        # Update biased first moment estimate
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        
        # Update biased second moment estimate
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads ** 2
        
        # Bias correction
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        
        # Update parameters
        return params - self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)
    
    def reset(self):
        self.m = None
        self.v = None
        self.t = 0

---

## 5. Modern Optimizers <a name="modern"></a>

### 5.1 AdamW (Adam with Weight Decay)

**Problem with Adam + L2 regularization**:
- L2 regularization: $\mathcal{L}(\theta) + \frac{\lambda}{2}||\theta||^2$
- Gradient: $g_t + \lambda \theta_{t-1}$
- When using adaptive learning rates, regularization gets scaled too!

**Solution**: Decouple weight decay from gradient

$$
\theta_t = \theta_{t-1} - \eta \left(\frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_{t-1}\right)
$$

**Key difference**: Weight decay applied **after** adaptive scaling

### Implementation

In [ ]:
class AdamW:
    """AdamW optimizer (Adam with decoupled weight decay)."""
    
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8, weight_decay=0.01):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.weight_decay = weight_decay
        self.m = None
        self.v = None
        self.t = 0
        
    def update(self, params, grads):
        if self.m is None:
            self.m = np.zeros_like(params)
            self.v = np.zeros_like(params)
        
        self.t += 1
        
        # Update biased first moment estimate
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        
        # Update biased second moment estimate
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads ** 2
        
        # Bias correction
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        
        # Update parameters with decoupled weight decay
        params_new = params - self.lr * (m_hat / (np.sqrt(v_hat) + self.epsilon) + 
                                          self.weight_decay * params)
        
        return params_new
    
    def reset(self):
        self.m = None
        self.v = None
        self.t = 0

---

## 6. Practical Comparison <a name="comparison"></a>

### 6.1 Optimization on Different Landscapes

In [ ]:
def optimize(optimizer, func, grad_func, x0, max_iter=200):
    """
    Run optimizer on a function.
    
    Returns:
        trajectory: List of parameter values
        losses: List of function values
    """
    optimizer.reset()
    x = x0.copy()
    trajectory = [x.copy()]
    losses = [func(x)]
    
    for i in range(max_iter):
        grad = grad_func(x)
        x = optimizer.update(x, grad)
        trajectory.append(x.copy())
        losses.append(func(x))
    
    return np.array(trajectory), np.array(losses)

### Compare on Rosenbrock Function

In [ ]:
# Starting point
x0 = np.array([-1.0, -1.0])

# Create optimizers
optimizers = {
    'SGD': SGD(lr=0.001),
    'Momentum': Momentum(lr=0.001, beta=0.9),
    'AdaGrad': AdaGrad(lr=0.1),
    'RMSprop': RMSprop(lr=0.01, beta=0.9),
    'Adam': Adam(lr=0.01),
    'AdamW': AdamW(lr=0.01, weight_decay=0.0)
}

# Run optimization
results = {}
for name, optimizer in optimizers.items():
    trajectory, losses = optimize(
        optimizer, 
        TestFunctions.rosenbrock,
        TestFunctions.rosenbrock_grad,
        x0,
        max_iter=500
    )
    results[name] = {'trajectory': trajectory, 'losses': losses}
    print(f"{name:12s}: Final loss = {losses[-1]:.6f}, Final x = [{trajectory[-1][0]:.4f}, {trajectory[-1][1]:.4f}]")

print("\nOptimum: x = [1, 1], f(x) = 0")

### Visualize Trajectories

In [ ]:
# Plot contours
X, Y, Z = plot_function_2d(
    TestFunctions.rosenbrock, 
    [-2, 2], 
    [-1, 3], 
    'Optimizer Trajectories on Rosenbrock Function',
    levels=np.logspace(-0.5, 3.5, 30)
)

# Plot trajectories
colors = plt.cm.Set1(np.linspace(0, 1, len(optimizers)))
for (name, result), color in zip(results.items(), colors):
    traj = result['trajectory']
    plt.plot(traj[:, 0], traj[:, 1], '-o', label=name, 
             color=color, markersize=3, linewidth=2, alpha=0.7)

# Mark start and optimum
plt.plot(x0[0], x0[1], 'k*', markersize=20, label='Start', zorder=10)
plt.plot(1, 1, 'r*', markersize=20, label='Optimum', zorder=10)

plt.legend(loc='upper left', fontsize=10)
plt.show()

### Plot Convergence

In [ ]:
plt.figure(figsize=(14, 5))

# Linear scale
plt.subplot(1, 2, 1)
for name, result in results.items():
    plt.plot(result['losses'], label=name, linewidth=2, alpha=0.8)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Convergence (Linear Scale)')
plt.legend()
plt.grid(True, alpha=0.3)

# Log scale
plt.subplot(1, 2, 2)
for name, result in results.items():
    plt.semilogy(result['losses'], label=name, linewidth=2, alpha=0.8)
plt.xlabel('Iteration')
plt.ylabel('Loss (log scale)')
plt.title('Convergence (Log Scale)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.2 Compare on Beale Function

In [ ]:
# Starting point for Beale
x0_beale = np.array([0.0, 0.0])

# Run optimization on Beale
results_beale = {}
for name, optimizer in optimizers.items():
    trajectory, losses = optimize(
        optimizer, 
        TestFunctions.beale,
        TestFunctions.beale_grad,
        x0_beale,
        max_iter=300
    )
    results_beale[name] = {'trajectory': trajectory, 'losses': losses}

# Plot
X, Y, Z = plot_function_2d(
    TestFunctions.beale, 
    [-4.5, 4.5], 
    [-4.5, 4.5], 
    'Optimizer Trajectories on Beale Function',
    levels=40
)

for (name, result), color in zip(results_beale.items(), colors):
    traj = result['trajectory']
    plt.plot(traj[:, 0], traj[:, 1], '-o', label=name, 
             color=color, markersize=3, linewidth=2, alpha=0.7)

plt.plot(x0_beale[0], x0_beale[1], 'k*', markersize=20, label='Start', zorder=10)
plt.plot(3, 0.5, 'r*', markersize=20, label='Optimum', zorder=10)
plt.legend(loc='upper left')
plt.show()

---

## 7. Hyperparameter Sensitivity <a name="sensitivity"></a>

### Learning Rate Sensitivity

In [ ]:
# Test different learning rates for Adam
learning_rates = [0.0001, 0.001, 0.01, 0.1, 1.0]
x0 = np.array([-1.0, -1.0])

plt.figure(figsize=(14, 5))

for lr in learning_rates:
    optimizer = Adam(lr=lr)
    trajectory, losses = optimize(
        optimizer,
        TestFunctions.rosenbrock,
        TestFunctions.rosenbrock_grad,
        x0,
        max_iter=500
    )
    plt.semilogy(losses, label=f'lr={lr}', linewidth=2)

plt.xlabel('Iteration')
plt.ylabel('Loss (log scale)')
plt.title('Adam: Learning Rate Sensitivity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Observations:")
print("- Too small (0.0001): Slow convergence")
print("- Too large (1.0): Divergence or instability")
print("- Sweet spot: 0.001 - 0.01")

### Momentum Coefficient Sensitivity

In [ ]:
# Test different momentum coefficients
betas = [0.0, 0.5, 0.9, 0.95, 0.99]

plt.figure(figsize=(14, 5))

for beta in betas:
    optimizer = Adam(lr=0.01, beta1=beta)
    trajectory, losses = optimize(
        optimizer,
        TestFunctions.rosenbrock,
        TestFunctions.rosenbrock_grad,
        x0,
        max_iter=500
    )
    plt.semilogy(losses, label=f'β₁={beta}', linewidth=2)

plt.xlabel('Iteration')
plt.ylabel('Loss (log scale)')
plt.title('Adam: Momentum Coefficient (β₁) Sensitivity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

## 8. Best Practices <a name="best-practices"></a>

### Quick Reference Table

| Optimizer | When to Use | Learning Rate | Key Hyperparameters |
|-----------|-------------|---------------|--------------------|
| **SGD** | Simple problems, baseline | 0.01 - 0.1 | - |
| **SGD + Momentum** | Ill-conditioned, ravines | 0.01 - 0.1 | β = 0.9 |
| **AdaGrad** | Sparse features, NLP | 0.01 | ε = 1e-8 |
| **RMSprop** | RNNs, non-stationary | 0.001 | β = 0.9 |
| **Adam** | **Default choice** | 0.001 | β₁=0.9, β₂=0.999 |
| **AdamW** | With regularization | 0.001 | decay=0.01 |

### General Guidelines

#### 1. Default Choice: Adam or AdamW
- Works well on most problems
- Minimal tuning required
- Use AdamW if you need weight decay

#### 2. Learning Rate Schedules

**Warmup**: Gradually increase learning rate
```python
lr_t = lr_max * min(1.0, t / warmup_steps)
```

**Cosine Annealing**:
```python
lr_t = lr_min + 0.5 * (lr_max - lr_min) * (1 + cos(π * t / T))
```

**Step Decay**:
```python
lr_t = lr_0 * γ^(t // step_size)
```

#### 3. Gradient Clipping

Prevent exploding gradients:
```python
grad_norm = ||g||
if grad_norm > threshold:
    g = g * threshold / grad_norm
```

#### 4. Batch Size Considerations

- **Small batch** (32-128): Noisy gradients, better generalization
- **Large batch** (256-1024): Stable gradients, faster training (with more GPUs)
- **Linear scaling rule**: If batch size × k, then lr × k

#### 5. When SGD Might Be Better

- Vision tasks (ResNet, etc.) with proper tuning
- When you have time to tune hyperparameters
- Sometimes better final generalization

### Summary Decision Tree

```
Need to choose optimizer?
│
├─ Quick prototype / research? → Adam (lr=0.001)
│
├─ Production model with regularization? → AdamW (lr=0.001, decay=0.01)
│
├─ RNN / LSTM? → RMSprop or Adam
│
├─ Vision (ResNet, etc.) with time to tune? → SGD + Momentum + Schedule
│
└─ Sparse features (NLP, recommendations)? → AdaGrad or Adam
```

### Key Takeaways

1. ✅ **Start with Adam**: Good default, minimal tuning
2. ✅ **Use learning rate warmup**: Especially for large models
3. ✅ **Monitor gradients**: Check for vanishing/exploding
4. ✅ **Clip gradients**: For RNNs and unstable training
5. ✅ **Try different learning rates**: Most important hyperparameter
6. ✅ **Use learning rate schedules**: Improve final performance

---

## References

1. **Ruder (2016)**: *An overview of gradient descent optimization algorithms*
2. **Kingma & Ba (2015)**: *Adam: A Method for Stochastic Optimization*
3. **Loshchilov & Hutter (2019)**: *Decoupled Weight Decay Regularization* (AdamW)
4. **Sutskever et al. (2013)**: *On the importance of initialization and momentum*

---

**Congratulations!** You now understand the mathematics and practice of optimization algorithms. 🎉
